# 01 — PyTorch baselines (FP32)

Runs:
- FP32 (CUDA)
- FP16 (CUDA)

Optionally without reduced bit depth.

In [1]:
from pathlib import Path
import sys, os

# ---- Path setup (adjust if your repo layout differs) ----
PROJECT_ROOT = Path("..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
# Purpose: Establish PyTorch FP32 baseline (and optional FP16 baseline) on ImageNet val.
import os
import json
from pathlib import Path
import pandas as pd

# project imports
from config import ExperimentConfig, with_overrides
from runner import run_experiment
from utils import load_runs, flatten_runs
from plots import (
    plot_metric_vs_input_bits,
    plot_tradeoff_with_pareto,
    plot_tradeoff_scatter,
)

pd.set_option("display.max_columns", 200)

In [3]:
# Baselines (PyTorch backend only)
base = ExperimentConfig(
    backend="pytorch",
    device="cuda",             
    batch_size=1,
    input_quant_bits=8,        
    seed=42,
    num_eval_batches=500
)


In [4]:
cfg32 = with_overrides(base, model_precision="fp32")

payload, tracker = run_experiment(cfg32, split="val", save_results_flag=True)

# quick peek
print(payload["run_id"], payload["status"], payload["results"].get("top1_acc"))

Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 70.00% | Top-5: 90.00% | Infer: 4.09 ms/batch
  Batch [50/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 3.91 ms/batch
  Batch [60/500] Top-1: 80.00% | Top-5: 93.33% | Infer: 3.52 ms/batch
  Batch [70/500] Top-1: 80.00% | Top-5: 95.00% | Infer: 3.45 ms/batch
  Batch [80/500] Top-1: 82.00% | Top-5: 94.00% | Infer: 3.42 ms/batch
  Batch [90/500] Top-1: 81.67% | Top-5: 95.00% | Infer: 3.52 ms/batch
  Batch [100/500] Top-1: 82.86% | Top-5: 95.71% | Infer: 3.46 ms/batch
  Batch [110/500] Top-1: 81.25% | Top-5: 95.00% | Infer: 3.46 ms/batch
  Batch [120/500] Top-1: 80.00% | Top-5: 95.56% | Infer: 3.40 ms/batch
  Batch [130/500] Top-1: 80.00% | Top-5: 96.00% | Infer: 3.34 ms/batch
  Batch [140/500] Top-1: 80.91% | Top-5: 96.36% | Infer: 3.31 ms/batch
  Batch [150/500] Top-1: 80.00% | Top-5: 95.83% | Infer: 3.25 ms/batch
  Batch [160/500] Top-1: 80.00% | T

In [5]:
# Always load from disk (the source of truth)
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter to just this notebook’s scope (pytorch + no input quant)
df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp32"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.backend",
    "cfg.device",
    "cfg.batch_size",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.device"])

,run_id,cfg.backend,cfg.device,cfg.batch_size,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
7,resnet18_pytorch_fp32_in8b_cuda_bs1,pytorch,cuda,1,fp32,8,78.297872,94.255319,4.726721,211.563171,470


In [6]:
cfg16 = with_overrides(base, model_precision="fp16")

payload, tracker = run_experiment(cfg16, split="val", save_results_flag=True)

# quick peek
print(payload["run_id"], payload["status"], payload["results"].get("top1_acc"))

Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 70.00% | Top-5: 90.00% | Infer: 2.80 ms/batch
  Batch [50/500] Top-1: 85.00% | Top-5: 95.00% | Infer: 3.00 ms/batch
  Batch [60/500] Top-1: 80.00% | Top-5: 93.33% | Infer: 4.50 ms/batch
  Batch [70/500] Top-1: 80.00% | Top-5: 95.00% | Infer: 6.92 ms/batch
  Batch [80/500] Top-1: 82.00% | Top-5: 94.00% | Infer: 8.55 ms/batch
  Batch [90/500] Top-1: 81.67% | Top-5: 95.00% | Infer: 9.46 ms/batch
  Batch [100/500] Top-1: 82.86% | Top-5: 95.71% | Infer: 10.17 ms/batch
  Batch [110/500] Top-1: 81.25% | Top-5: 95.00% | Infer: 10.54 ms/batch
  Batch [120/500] Top-1: 80.00% | Top-5: 95.56% | Infer: 10.98 ms/batch
  Batch [130/500] Top-1: 80.00% | Top-5: 96.00% | Infer: 10.50 ms/batch
  Batch [140/500] Top-1: 80.91% | Top-5: 96.36% | Infer: 9.87 ms/batch
  Batch [150/500] Top-1: 80.00% | Top-5: 95.83% | Infer: 9.34 ms/batch
  Batch [160/500] Top-1: 80.00%

In [7]:
# Always load from disk (the source of truth)
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter to just this notebook’s scope (pytorch + no input quant)
df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp16"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.backend",
    "cfg.device",
    "cfg.batch_size",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.device"])

,run_id,cfg.backend,cfg.device,cfg.batch_size,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
3,resnet18_pytorch_fp16_in8b_cuda_bs1,pytorch,cuda,1,fp16,8,78.297872,94.255319,6.524806,153.261258,470


In [8]:
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp32", "fp16"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.model_precision",
    "res.infer_ms_avg",
    "res.infer_ms_min",
    "res.infer_ms_max",
]].sort_values("cfg.model_precision")

,run_id,cfg.model_precision,res.infer_ms_avg,res.infer_ms_min,res.infer_ms_max
3,resnet18_pytorch_fp16_in8b_cuda_bs1,fp16,6.524806,1.329808,20.289511
7,resnet18_pytorch_fp32_in8b_cuda_bs1,fp32,4.726721,1.436842,28.306499
